# 06.5 Dynamic topic modeling con BERTopic (Híbrido)

Este cuaderno prepara un flujo para modelar temas dinámicos sobre los posteos de Revista Anfibia usando una arquitectura híbrida:
- **Embeddings/Clustering:** Usa el texto original para capturar la sintaxis y semántica.
- **Representación (Palabras clave):** Usa el texto lematizado para evitar redundancia morfológica en español.

In [1]:
# EJECUTAR ESTA CELDA PRIMERO QUE NADA (Y REINICIAR EL KERNEL ANTES SI FALLÓ)
import coverage.types as ct

# Parche de compatibilidad para evitar que Numba/UMAP crasheen
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

## 1. Configuración del entorno

In [2]:
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
STOPWORDS_PATH = PROJECT_ROOT / "data" / "processed" / "stopwords_anfibia.txt"

assert DATA_PATH.exists(), f"Falta archivo de entrada: {DATA_PATH.name}"
assert STOPWORDS_PATH.exists(), f"Falta archivo de stopwords: {STOPWORDS_PATH.name}"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_hibrido"
GRAFICOS_DIR = OUTPUT_DIR / "graficos"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
GRAFICOS_DIR.mkdir(exist_ok=True, parents=True)

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga de Corpus y Stopwords
Preparamos los textos, normalizamos categorías y cargamos la lista de stopwords.

In [3]:
MAX_DOCUMENTS = 4000 ## el corpus igual tiene ~3600
USE_COLUMNS = [
    "doc_id", 
    "ID", "source_url", "title_text", "tagline", "autor_names",
    "post_date", "category", "post_tag", "text_type",
    "document_text_original", "document_text_final"
]

df = pd.read_csv(DATA_PATH, usecols=USE_COLUMNS, nrows=MAX_DOCUMENTS)

print(f"Documentos listos: {df.shape[0]}")

# Normalizar categorías vectorizado
def split_categories(val):
    return [p.strip() for p in re.split(r"[;,]", str(val)) if p.strip()]

categories_df = (
    df[["doc_id", "category"]]
    .assign(category_normalized=df["category"].map(split_categories))
    .explode("category_normalized")
    .fillna({"category_normalized": "Sin categoria"})
    [["doc_id", "category_normalized"]]
)

# Textos, fechas y stopwords
docs_naturales = df["document_text_original"].tolist()
docs_lematizados = df["document_text_final"].tolist()
timestamps = pd.to_datetime(df["post_date"].str.split(' ').str[0], format="%Y-%m-%d", errors="coerce")

with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    STOPWORDS_LIST = f.read().splitlines()

Documentos listos: 3555


## 3. Entrenamiento Híbrido y Reducción de Outliers (Instancia Única)

1. Calculamos embeddings sobre el texto.
2. Clusterizamos (UMAP + HDBSCAN).
3. Reasignamos las notas outliers (-1).
4. Extraemos las palabras clave con el texto lematizado y las stopwords.

In [4]:
print("1. Calculando embeddings originales...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings_naturales = embedding_model.encode(docs_naturales, show_progress_bar=True)

print("\n2. Entrenando modelo base (HDBSCAN puro)...")
vectorizer_model = CountVectorizer(stop_words=STOPWORDS_LIST, min_df=3)
representation_model = KeyBERTInspired()
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

topic_model = BERTopic(
    umap_model=umap_model,
    embedding_model=embedding_model,
    language="spanish",
    calculate_probabilities=True,
    min_topic_size=10,
    nr_topics=None,
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
)

topics_base, probs = topic_model.fit_transform(docs_lematizados, embeddings=embeddings_naturales)

print("\n3. Reasignando notas outliers (-1)...")
topics_reducidos = topic_model.reduce_outliers(
    docs_lematizados,
    topics_base,
    strategy="embeddings",
    embeddings=embeddings_naturales
)

print("\n4. Recalibrando palabras clave lematizadas de los tópicos (c-TF-IDF)...")
# Hay que pasarle el vectorizador de nuevo sí o sí
topic_model.update_topics(
    docs_lematizados, 
    topics=topics_reducidos,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

topic_model.save(OUTPUT_DIR / "bertopic_hibrido_model") 
topics = topics_reducidos
print("\nModelo optimizado y guardado exitosamente.")

1. Calculando embeddings originales...


Batches: 100%|██████████| 112/112 [01:34<00:00,  1.19it/s]



2. Entrenando modelo base (HDBSCAN puro)...


2026-09-09 11:04:05,104 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



3. Reasignando notas outliers (-1)...

4. Recalibrando palabras clave lematizadas de los tópicos (c-TF-IDF)...


2026-09-09 11:04:22,541 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.



Modelo optimizado y guardado exitosamente.


## 4. Diagnóstico visual de Outliers
Comparamos cuánto ruido generó el modelo base y cómo quedó tras la reasignación.

In [5]:
outliers_base = topics_base.count(-1)
clasificados_base = len(topics_base) - outliers_base
outliers_red = topics_reducidos.count(-1)
clasificados_red = len(topics_reducidos) - outliers_red

fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=[
        f"Puro",
        f"Post-Reducción"
    ]
)

fig.add_trace(go.Pie(labels=["Ruido (-1)", "Válidos"], values=[outliers_base, clasificados_base], hole=0.4, marker=dict(colors=["#d62728", "#2ca02c"])), row=1, col=1)
fig.add_trace(go.Pie(labels=["Ruido (-1)", "Válidos"], values=[outliers_red, clasificados_red], hole=0.4, marker=dict(colors=["#d62728", "#2ca02c"])), row=1, col=2)

fig.update_layout(title_text="<b>Comparacion de topicos</b>", template="plotly_white", height=450, width=800)
fig.show()

## 5. Extracción de Evolución Temporal y Gráficos Básicos

In [6]:
print("Calculando evolución temporal de los tópicos...")
topics_over_time = topic_model.topics_over_time(
    docs=docs_lematizados,
    timestamps=timestamps.dt.to_pydatetime().tolist(),
    nr_bins=20,
)
topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time.csv", index=False)

# 1. Asignar al df original 
df['topic'] = topics

# # 2. Armar analysis_df ya con el tópico incluido, y luego hacer el merge
# analysis_df = df[['doc_id', 'post_date', 'topic']].copy()
# analysis_df = analysis_df.merge(categories_df, on='doc_id', how='left')

# analysis_df['post_date'] = pd.to_datetime(analysis_df['post_date'], errors='coerce')

# topics_by_category = (
#     analysis_df
#     .dropna(subset=['post_date'])
#     .assign(date=lambda x: x['post_date'].dt.to_period('M').dt.to_timestamp())
#     .groupby(['date', 'category_normalized', 'topic'])
#     .size()
#     .reset_index(name='count')
# )
# topics_by_category.to_csv(OUTPUT_DIR / 'topics_over_time_by_category.csv', index=False)
# print("-> Archivos temporales guardados en outputs.")

fig = topic_model.visualize_barchart(top_n_topics=16, n_words=8, title="<b>Palabras representativas (Híbrido)</b>")
colors = px.colors.qualitative.G10
for i, trace in enumerate(fig.data):
    trace.marker.color = colors[i % len(colors)]

fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
fig.update_layout(height=850, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

fig.write_html(GRAFICOS_DIR / "01_barchart_topicos.html")
fig.show()

Calculando evolución temporal de los tópicos...


## 6. Diagnóstico: Coherencia Semántica (NPMI)

Para validar si las palabras clave que eligió el modelo para un tópico tienen sentido juntas o son una ensalada de términos inconexos, calculamos la **Coherencia NPMI** (*Normalized Pointwise Mutual Information*).

### ¿Cómo funciona la métrica?

Primero se calcula el **PMI** tradicional, que compara qué tan probable es que dos palabras aparezcan juntas en la misma nota respecto a que aparezcan separadas de pura casualidad:

$$PMI(w_i, w_j) = \log_2 \left( \frac{P(w_i, w_j)}{P(w_i) \cdot P(w_j)} \right)$$

* $P(w_i, w_j)$: proporción de notas donde coocurren ambas palabras.
* $P(w_i) \cdot P(w_j)$: probabilidad esperada si fueran independientes.

El problema del PMI es que no está acotado y se distorsiona con palabras muy poco frecuentes. Para solucionarlo, se normaliza dividiendo por su auto-información conjunta ($-\log_2 P(w_i, w_j)$), obteniendo el **NPMI**:

$$NPMI(w_i, w_j) = \frac{\log_2 \left( \frac{P(w_i, w_j)}{P(w_i) \cdot P(w_j)} \right)}{-\log_2 P(w_i, w_j)}$$

El valor final del tópico es el promedio del NPMI entre todos los pares de su Top 10 de términos representativos:

$$C_{NPMI}(T) = \frac{2}{N(N-1)} \sum_{i < j} NPMI(w_i, w_j)$$

### Criterio de lectura (escala fija de -1 a +1):

* **$+1$ (Coocurrencia total):** Siempre que aparece una palabra, aparece la otra.
* **$0$ (Independencia):** Aparecen juntas lo esperable por puro azar.
* **$-1$ (Exclusión mutua):** Nunca jamás comparten un artículo.

**Diagnóstico:**
* **$> 0.10$ (Alta coherencia):** Las palabras clave forman un núcleo temático sólido y claro.
* **$0.00$ a $0.10$ (Aceptable / Amplio):** Tópico interpretable pero de agenda generalista.
* **$< -0.05$ (Ruidoso):** Términos descolgados. Candidato a revisar, fusionar o descartar.

In [7]:
# Reutilizamos el vectorizador ya entrenado por BERTopic
vectorizer = topic_model.vectorizer_model
X = vectorizer.transform(docs_lematizados)
binary_matrix = (X > 0).astype(int)

doc_counts = np.array(binary_matrix.sum(axis=0)).flatten()
n_docs = binary_matrix.shape[0]

def calculate_topic_npmi(words, top_n=10):
    words = words[:top_n]
    vocab = vectorizer.vocabulary_
    indices = [vocab[w] for w in words if w in vocab]
    
    if len(indices) < 2: 
        return 0.0
    
    npmi_scores = []
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            idx_i, idx_j = indices[i], indices[j]
            p_i = doc_counts[idx_i] / n_docs
            p_j = doc_counts[idx_j] / n_docs
            
            # Coocurrencia directa sobre matriz sparse sin toarray()
            cooc = binary_matrix[:, idx_i].T.dot(binary_matrix[:, idx_j])[0, 0]
            p_ij = cooc / n_docs
            
            if p_ij > 0:
                npmi_scores.append((np.log2(p_ij / (p_i * p_j))) / (-np.log2(p_ij)))
            else:
                npmi_scores.append(-1.0)
                
    return np.mean(npmi_scores) if npmi_scores else 0.0

# Obtenemos el conteo global una sola vez para no llamar get_topic_info en loop
counts_dict = topic_model.get_topic_info().set_index("Topic")["Count"].to_dict()

topic_metrics = []
for t_id, words_weights in topic_model.get_topics().items():
    if t_id == -1: 
        continue
    words = [w for w, _ in words_weights]
    count = counts_dict.get(t_id, 0)
    topic_metrics.append({
        "Topic": t_id,
        "Name": ", ".join(words[:4]),
        "Docs": count,
        "% Corpus": round((count / n_docs) * 100, 2),
        "Coherencia_NPMI": round(calculate_topic_npmi(words), 4)
    })

metrics_df = pd.DataFrame(topic_metrics).sort_values(by="Docs", ascending=False)
display(metrics_df.head(15))

,Topic,Name,Docs,% Corpus,Coherencia_NPMI
1,1,"político, política, electoral, candidato",258,7.26,0.3154
3,3,"literario, literatura, escritora, autor",254,7.14,0.2763
0,0,"feminismo, feminista, patriarcado, masculinidad",218,6.13,0.2868
2,2,"fifa, fútbol, futbolista, deporte",155,4.36,0.3157
4,4,"pedagógico, enseñanza, escolar, educativo",141,3.97,0.2686
5,5,"déficit, deuda, endeudamiento, inflación",140,3.94,0.3114
12,12,"autoritario, dictadura, democracia, político",126,3.54,0.2078
30,30,"muerto, maldonado, muerte, morir",117,3.29,0.1757
7,7,"mediático, periodístico, periodista, noticia",110,3.09,0.1086
23,23,"político, política, sociedad, gobierno",95,2.67,0.1156


Graficamos 

In [8]:
# 1. Tabla de tópicos más amplios (con criterio de representatividad)
# Consideramos "amplios" a los que superan el 1.0% del corpus o el Top 10 por volumen
topicos_amplios = metrics_df[metrics_df["% Corpus"] >= 1.0].copy()

# Agregamos etiqueta de diagnóstico semántico
def diagnostico_npmi(npmi):
    if npmi >= 0.10:
        return "Alta coherencia"
    elif npmi >= 0.00:
        return "Aceptable / Amplio"
    else:
        return "Ruidoso"

topicos_amplios["Diagnostico"] = topicos_amplios["Coherencia_NPMI"].map(diagnostico_npmi)

# print(f"=== Tópicos más amplios (>= 1.0% del corpus): {len(topicos_amplios)} tópicos ===")
# display(topicos_amplios)

# 2. Gráfico de Tópicos más Amplios vs. Coherencia NPMI
fig_amplios = px.bar(
    topicos_amplios.sort_values(by="Docs", ascending=True),
    x="Docs",
    y="Name",
    orientation="h",
    color="Coherencia_NPMI",
    color_continuous_scale="RdYlGn",
    range_color=[-0.1, 0.25],
    custom_data=["Topic", "% Corpus", "Coherencia_NPMI", "Diagnostico"]
)

fig_amplios.update_traces(
    hovertemplate=(
        "<b>Tópico %{customdata[0]}:</b> %{y}<br>"
        "<b>Volumen:</b> %{x} notas (%{customdata[1]}% del corpus)<br>"
        "<b>NPMI:</b> %{customdata[2]:.4f} (%{customdata[3]})"
        "<extra></extra>"
    )
)

fig_amplios.update_layout(
    title="<b>Tópicos más amplios del corpus y su Coherencia Semántica</b>",
    xaxis_title="Cantidad de notas",
    yaxis_title="",
    height=max(400, len(topicos_amplios) * 32),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    coloraxis_colorbar=dict(title="Coherencia<br>NPMI")
)
fig_amplios.update_xaxes(showgrid=True, gridcolor="#ebebeb")

fig_amplios.write_html(GRAFICOS_DIR / "04_topicos_mas_amplios.html")
fig_amplios.show()

## 7. Articulos mas representativos de los topicos
Calculamos el centroide de cada tópico sobre la matriz de embeddings y extraemos los 3 artículos con mayor similitud coseno respecto a dicho centroide.

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# PARÁMETROS DE VISUALIZACIÓN
TOP_N_NOTAS = 1         # Cantidad de artículos representativos a extraer por tópico
TOP_N_TOPICOS = 10       # Cantidad de tópicos a mostrar en el display final (None para mostrar todos)


# 2. Asegurar columnas clave
df["topic_assigned"] = topics
df["post_date_dt"] = pd.to_datetime(df["post_date"], errors="coerce")

# 3. Tabla de notas más representativas por tópico global
topicos_info = topic_model.get_topic_info()
filas_representativas = []

for _, row in topicos_info.iterrows():
    t_id = row["Topic"]
    if t_id == -1:
        continue
        
    t_name = row["Name"]
    indices_topico = df[df["topic_assigned"] == t_id].index.tolist()
    
    if not indices_topico:
        continue
        
    # Centroide semántico del tópico
    centroide = embeddings_naturales[indices_topico].mean(axis=0, keepdims=True)
    
    # Similitud coseno respecto al centroide
    similitudes = cosine_similarity(embeddings_naturales[indices_topico], centroide).flatten()
    
    # Top N artículos más representativos
    top_locales = np.argsort(similitudes)[::-1][:TOP_N_NOTAS]
    
    for rank, idx_local in enumerate(top_locales, start=1):
        idx_global = indices_topico[idx_local]
        nota = df.loc[idx_global]
        filas_representativas.append({
            "Topic": t_id,
            "Topic_Name": t_name,
            "Rank": rank,
            "Similitud_Centroide": round(float(similitudes[idx_local]), 4),
            "Titulo": nota.get("title_text", "Sin título"),
            "Fecha": str(nota.get("post_date", ""))[:10],
            "Tagline": str(nota.get("tagline", ""))[:80] if pd.notna(nota.get("tagline")) else "",
            "Tag": str(nota.get("post_tag", "")).split(";")[0].strip() if pd.notna(nota.get("post_tag")) else ""
        })

df_representativos_global = pd.DataFrame(filas_representativas)

# Vista previa controlada por variables
if TOP_N_TOPICOS is not None:
    topicos_unicos = df_representativos_global["Topic"].unique()[:TOP_N_TOPICOS]
    display(df_representativos_global[df_representativos_global["Topic"].isin(topicos_unicos)])
else:
    display(df_representativos_global)

,Topic,Topic_Name,Rank,Similitud_Centroide,Titulo,Fecha,Tagline,Tag
0,0,0_feminismo_feminista_patriarcado_masculinidad,1,0.8536,"Ni machos, ni fachos",2017-04-12,Una agenda feminista para la Justicia,#NiUnaMenos
1,1,1_político_política_electoral_candidato,1,0.8757,El arte de capar monos,2016-05-19,"No es ""ceocracia"", es política",
2,2,2_fifa_fútbol_futbolista_deporte,1,0.8362,Polaroids de locura ordinaria,2018-03-21,Bilardo después de Bilardo,
3,3,3_literario_literatura_escritora_autor,1,0.8358,Todos anfibios,2015-06-11,Al rescate,
4,4,4_pedagógico_enseñanza_escolar_educativo,1,0.8851,La autoridad pedagógica en jaque,2020-11-18,Soledad Acuña contra la escuela pública,
5,5,5_déficit_deuda_endeudamiento_inflación,1,0.9185,"Regalos para los grandes, crisis para el resto",2024-05-30,Dictamen favorable para la Ley Bases,Derechas
6,6,6_aborto_abortar_anticonceptivo_embarazo,1,0.9114,No es obligatorio abortar,2018-03-05,La diputada de Cambiemos que apoya la ley,
7,7,7_mediático_periodístico_periodista_noticia,1,0.8560,La movilidad social por otros medios,2024-04-05,Gran Hermano Tucumán,Gran Hermano
8,8,8_político_política_presidencial_libertario,1,0.9040,Fingir demencia,2024-04-29,La salud mental del presidente,Derechas
9,9,9_epidemia_pandemia_vacunar_virus,1,0.8859,Sabía que ibas a reaccionar así,2020-05-19,Cómo nos comportamos ante la flexibilización,


## 8. Topicos en el tiempo

Para interpretar los picos de cada topico, identificamos sus notas más representativas siguiendo esta lógica:

1. **Ventana Temporal Exacta (Bin Dinámico):** Delimitamos el intervalo temporal exacto ($[t_{\text{inicio}}, t_{\text{fin}}]$) generado por el binning de `topics_over_time` donde ocurrió el récord de frecuencia ($\max(\text{Frequency})$).
2. **Centroide Local del Pico:** Calculamos el **centroide semántico local** ($\vec{C}_{\text{pico}}$) promediando únicamente los embeddings de las notas publicadas durante esa cresta temporal:
   $$\vec{C}_{\text{pico}} = \frac{1}{|D_{\text{pico}}|} \sum_{d \in D_{\text{pico}}} \vec{v}_d$$
3. **Ranking por Similitud Coseno:** Evaluamos la distancia angular de cada artículo del intervalo contra dicho centroide coyuntural:
   $$\text{Similitud}(\vec{v}_{\text{nota}}, \vec{C}_{\text{pico}}) = \frac{\vec{v}_{\text{nota}} \cdot \vec{C}_{\text{pico}}}{\|\vec{v}_{\text{nota}}\| \|\vec{C}_{\text{pico}}\|}$$
4. **Artículos Más Representativos:** Extraemos el Top $N$ de notas más cercanas al núcleo del evento, reflejando fielmente la cobertura periodística concreta que generó el pico de publicaciones.


Pero muchos topicos no tienen solo su pico maximo:

1. **Detección de Picos Locales y Máximos:** Escaneamos la serie temporal de cada tópico para identificar:
   * **Máximo Histórico:** El punto absoluto con mayor volumen de publicaciones ($\max(\text{Frequency})$).
   * **Picos Locales:** Momentos donde la frecuencia supera a sus períodos adyacentes ($\text{Freq}_t > \text{Freq}_{t-1}$ y $\text{Freq}_t > \text{Freq}_{t+1}$) y sobrepasa el umbral mínimo de actividad ($\text{Freq}_t \ge \max(3, \overline{\text{Freq}})$).
2. **Ventana Temporal Dinámica:** Delimitamos el intervalo exacto $[t_{\text{inicio}}, t_{\text{fin}}]$ correspondiente a cada bin temporal para aislar únicamente las notas publicadas en ese período.
3. **Centroide Semántico Local:** Calculamos el vector promedio coyuntural ($\vec{C}_{\text{bin}}$) a partir de los embeddings de las notas de esa ventana:
   $$\vec{C}_{\text{bin}} = \frac{1}{|D_{\text{bin}}|} \sum_{d \in D_{\text{bin}}} \vec{v}_d$$
4. **Artículos Clave por Período:** 
   * En **meses habituales**, extraemos la nota más cercana a $\vec{C}_{\text{bin}}$ para contextualizar la línea continua.
   * En **picos locales y máximos**, seleccionamos el Top 3 de notas más representativas para desplegar el desglose editorial que motivó el salto de cobertura.

In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ==============================================================================
# PARÁMETROS
# ==============================================================================
TOP_N_NOTAS_PICO = 3         # Cantidad de artículos si el mes es un pico
TOP_N_NOTAS_NORMAL = 1       # Cantidad de artículos para meses normales
MIN_FREQ_PICO = 3            # Frecuencia mínima absoluta para considerar un pico


df["post_date_dt"] = pd.to_datetime(df["post_date"], errors="coerce")
unique_timestamps = sorted(pd.to_datetime(topics_over_time["Timestamp"].unique()))

top_topics_ids = topic_model.get_topic_info().query("Topic != -1")["Topic"].tolist()

# Mapeo seguro de nombres de tópicos usando el modelo entrenado
topic_names_dict = {}
for t_id in top_topics_ids:
    words_weights = topic_model.get_topic(t_id)
    if words_weights:
        topic_names_dict[t_id] = f"{t_id}_" + "_".join([w for w, _ in words_weights[:3]])
    else:
        topic_names_dict[t_id] = f"Topic_{t_id}"

timeline_data = []

print(f"Analizando serie temporal y picos para {len(top_topics_ids)} tópicos...")

for t_id in top_topics_ids:
    # Ordenar cronológicamente
    df_t = topics_over_time[topics_over_time["Topic"] == t_id].sort_values("Timestamp").reset_index(drop=True)
    if df_t.empty:
        continue
    
    # Asignación segura del nombre desde el diccionario (evita KeyError: 'Name')
    t_name = topic_names_dict.get(t_id, f"Topic_{t_id}")
    freqs = df_t["Frequency"].values
    
    # --- Detección de múltiples picos ---
    '''
    Un punto no califica como pico si tiene menos de 3 notas o si no supera el promedio de publicaciones de ese tópico.
    '''
    umbral = max(MIN_FREQ_PICO, freqs.mean())
    
    if len(freqs) > 2:
        mayor_al_anterior = np.r_[False, freqs[1:] > freqs[:-1]]
        mayor_al_siguiente = np.r_[freqs[:-1] > freqs[1:], False]
        picos_locales = mayor_al_anterior & mayor_al_siguiente & (freqs >= umbral)
        picos_idx = set(np.where(picos_locales)[0])
    else:
        picos_idx = set()
        
    if len(freqs) > 0:
        picos_idx.add(np.argmax(freqs)) # Asegurar siempre el máximo absoluto
    
    for i, row in df_t.iterrows():
        fecha_str = row["Timestamp"]
        fecha_dt = pd.to_datetime(fecha_str)
        freq = int(row["Frequency"])
        
        is_peak = (i in picos_idx)
        is_max = (i == np.argmax(freqs))
        
        # Filtro temporal exacto del bin
        idx_bin = unique_timestamps.index(fecha_dt)
        t_start = unique_timestamps[idx_bin - 1] if idx_bin > 0 else fecha_dt - pd.DateOffset(months=3)
        t_end = unique_timestamps[idx_bin + 1] if idx_bin < len(unique_timestamps) - 1 else fecha_dt + pd.DateOffset(months=3)
        
        mask_bin = (df["topic"] == t_id) & (df["post_date_dt"] >= t_start) & (df["post_date_dt"] <= t_end)
        indices_bin = df[mask_bin].index.tolist()
        
        n_necesarias = TOP_N_NOTAS_PICO if is_peak else TOP_N_NOTAS_NORMAL
        if len(indices_bin) < n_necesarias:
            mask_año = (df["topic"] == t_id) & (df["post_date_dt"].dt.year == fecha_dt.year)
            indices_bin = df[mask_año].index.tolist()
            
        notas_list = []
        if indices_bin:
            # Centroide local del bin
            ref_vector = embeddings_naturales[indices_bin].mean(axis=0, keepdims=True)
            similitudes = cosine_similarity(embeddings_naturales[indices_bin], ref_vector).flatten()
            top_locales = np.argsort(similitudes)[::-1][:n_necesarias]
            
            for rank, idx_local in enumerate(top_locales, start=1):
                idx_global = indices_bin[idx_local]
                nota = df.loc[idx_global]
                notas_list.append({
                    "Titulo": nota.get("title_text", "Sin título"),
                    "Tagline": str(nota.get("tagline", ""))[:80] if pd.notna(nota.get("tagline")) else "",
                    "Tag": str(nota.get("post_tag", "")).split(";")[0].strip() if pd.notna(nota.get("post_tag")) else ""
                })
                
        timeline_data.append({
            "Topic": t_id,
            "Name": t_name,
            "Fecha": fecha_dt,
            "Frecuencia": freq,
            "Es_Pico": is_peak,
            "Es_Maximo": is_max,
            "Articulos": notas_list
        })

df_timeline = pd.DataFrame(timeline_data)
display(df_timeline[df_timeline["Es_Pico"] == True][["Topic", "Name", "Fecha", "Frecuencia", "Es_Maximo"]].head(15))

Analizando serie temporal y picos para 46 tópicos...


,Topic,Name,Fecha,Frecuencia,Es_Maximo
4,0,0_feminismo_feminista_patriarcado,2015-03-09 04:48:00,15,False
8,0,0_feminismo_feminista_patriarcado,2018-01-01 09:36:00,30,True
12,0,0_feminismo_feminista_patriarcado,2020-10-26 14:24:00,15,False
23,1,1_político_política_electoral,2015-03-09 04:48:00,24,False
26,1,1_político_política_electoral,2017-04-19 02:24:00,33,True
29,1,1_político_política_electoral,2019-05-31 00:00:00,18,False
32,1,1_político_política_electoral,2021-07-10 21:36:00,22,False
36,1,1_político_política_electoral,2024-05-05 02:24:00,18,False
42,2,2_fifa_fútbol_futbolista,2014-06-24 21:36:00,30,True
45,2,2_fifa_fútbol_futbolista,2016-08-04 19:12:00,10,False


Graficamos

In [11]:
TOP_N_TRAZAS_ACTIVAS = 3 

# Mapeo directo para consultar rápido (O(1)) por Tópico y Fecha
timeline_dict = {(row["Topic"], row["Fecha"]): row for row in timeline_data}

fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=len(top_topics_ids))
colors = px.colors.qualitative.G10

rombos_picos = []

for i, t in enumerate(fig.data):
    c = colors[i % len(colors)]
    vis = True if i < TOP_N_TRAZAS_ACTIVAS else 'legendonly'
    
    raw_name = t.name
    try:
        topic_id = int(raw_name.split("_")[0])
    except (ValueError, IndexError):
        continue

    clean_name = raw_name.replace("_", " ")

    textos_hover_linea = []
    
    for x_val, y_val in zip(t.x, t.y):
        fecha_dt = pd.to_datetime(x_val)
        info_punto = timeline_dict.get((topic_id, fecha_dt))
        
        if info_punto and info_punto["Articulos"]:
            # Si el punto es un pico (local o máximo), armamos el hover enriquecido (Top 3)
            if info_punto["Es_Pico"]:
                detalles_hover = ""
                for n in info_punto["Articulos"]:
                    titulo = n["Titulo"][:39] + ("..." if len(n["Titulo"]) > 42 else "")
                    tagline_str = f"<br>&nbsp;&nbsp;&nbsp;&nbsp;<i>\"{n['Tagline'][:72]}...\"</i>" if n["Tagline"] and n["Tagline"].lower() != "nan" else ""
                    tag_str = f" <b>(#{n['Tag']})</b>" if n["Tag"] and n["Tag"].lower() != "nan" else ""
                    detalles_hover += f"<br>• <b>{titulo}</b>{tag_str}{tagline_str}"
                
                label_pico = "Máximo Histórico" if info_punto["Es_Maximo"] else "Pico Destacado"
                textos_hover_linea.append(
                    f"<br><b>[{label_pico}]</b><br><b>Artículos representativos:</b>{detalles_hover}"
                )
            else:
                # Mes normal: solo la nota clave individual
                tit = info_punto["Articulos"][0]["Titulo"]
                tit_corto = (tit[:45] + "...") if len(tit) > 48 else tit
                textos_hover_linea.append(f"<br>• <b>Nota del mes:</b> {tit_corto}")
        else:
            textos_hover_linea.append("")

        # Dibujar el rombo visual ÚNICAMENTE en el Máximo Histórico global
        if info_punto and info_punto["Es_Maximo"]:
            detalles_rombo = ""
            for n in info_punto["Articulos"]:
                titulo = n["Titulo"][:39] + ("..." if len(n["Titulo"]) > 42 else "")
                tagline_str = f"<br>&nbsp;&nbsp;&nbsp;&nbsp;<i>\"{n['Tagline'][:72]}...\"</i>" if n["Tagline"] and n["Tagline"].lower() != "nan" else ""
                tag_str = f" <b>(#{n['Tag']})</b>" if n["Tag"] and n["Tag"].lower() != "nan" else ""
                detalles_rombo += f"<br>• <b>{titulo}</b>{tag_str}{tagline_str}"
                
            if not info_punto["Articulos"]:
                detalles_rombo = "<br>• Sin notas exactas en el período"

            rombos_picos.append(
                go.Scatter(
                    x=[fecha_dt], 
                    y=[info_punto["Frecuencia"]], 
                    mode="markers", 
                    showlegend=False, 
                    visible=vis, 
                    legendgroup=f"g{topic_id}",
                    marker=dict(size=12, color=c, symbol="diamond", line=dict(width=1.5, color="#111")),
                    hovertemplate=(
                        f"<b> Máximo Histórico: {clean_name}</b><br>"
                        "Fecha: %{x|%Y-%m}<br>"
                        "Frecuencia: %{y}<br>"
                        f"<b>Artículos representativos:</b>{detalles_rombo}"
                        "<extra></extra>"
                    )
                )
            )

    # Actualizar la línea continua aplicando el customdata unificado
    t.update(
        name=clean_name, 
        visible=vis, 
        legendgroup=f"g{topic_id}", 
        line=dict(color=c, width=2.5),
        customdata=textos_hover_linea,
        hovertemplate=(
            f"<b>{clean_name}</b><br>"
            "Fecha: %{x|%Y-%m}<br>"
            "Frecuencia: %{y}"
            "%{customdata}<extra></extra>"
        )
    )

fig.add_traces(rombos_picos)

fig.update_layout(
    title="<b>Evolución temporal y Coyuntura Periodística</b>", 
    plot_bgcolor='rgba(0,0,0,0)', 
    paper_bgcolor='#fff', 
    height=750, 
    hovermode="closest",
    hoverlabel=dict(bgcolor="white", font_size=16)
)
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")
fig.update_xaxes(showgrid=True, gridcolor="#ebebeb")

fig.write_html(GRAFICOS_DIR / "02_evolucion_topicos_completa.html")
fig.show()

## 9. Proyección 2D del Espacio Semántico

In [ ]:
import plotly.express as px
import pandas as pd
from umap import UMAP

# ==============================================================================
# PROYECCIÓN 2D DEL ESPACIO SEMÁNTICO (LIMPIO Y FUNCIONAL)
# ==============================================================================
reducer = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
coords_2d = reducer.fit_transform(embeddings_naturales)

topic_info = topic_model.get_topic_info()
topic_names = {row['Topic']: row['Name'] for _, row in topic_info.iterrows()}

In [14]:
# 1. Limpieza de taglines para evitar errores en el hover
clean_taglines = (
    df['tagline']
    .fillna("")
    .astype(str)
    .apply(lambda x: (x[:80] + "...") if len(x) > 83 else x)
    .replace("", "Sin bajada")
    .values
)

# 2. DataFrame base para visualización
plot_df = pd.DataFrame({
    'x': coords_2d[:, 0], 
    'y': coords_2d[:, 1],
    'topic': topics, 
    'title': df['title_text'].values,
    'tagline': clean_taglines
})

# FILTRO CLAVE: Excluir los outliers (-1) para que el gráfico no se sature de ruido gris
plot_df = plot_df[plot_df['topic'] != -1].copy()

# Mapeo de nombres limpios para la leyenda
plot_df['topic_name'] = plot_df['topic'].map(topic_names)
plot_df['topic_label'] = plot_df['topic'].astype(str) + ": " + plot_df['topic_name']

# Ordenar categorías estrictamente por ID de tópico positivo
topics_ordenados_validos = sorted(plot_df['topic'].unique())
categorias_ordenadas = [f"{t}: {topic_names[t]}" for t in topics_ordenados_validos]
plot_df['topic_label'] = pd.Categorical(plot_df['topic_label'], categories=categorias_ordenadas, ordered=True)

# 3. Gráfico de dispersión interactivo
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_label', 
    custom_data=['topic_label', 'title', 'tagline']
)

fig.update_traces(
    hovertemplate=(
        "<b>Tópico:</b> %{customdata[0]}<br>"
        "<b>Título:</b> %{customdata[1]}<br>"
        "<i>%{customdata[2]}</i>"
        "<extra></extra>"
    ), 
    marker=dict(size=6, opacity=0.8)
)

fig.update_layout(
    title="<b>Espacio de documentos (Embeddings Híbridos)</b>", 
    plot_bgcolor='rgba(0,0,0,0)', 
    paper_bgcolor='#fff', 
    height=700,
    hovermode="closest",
    xaxis=dict(showticklabels=False, zeroline=False, showgrid=False, title=""),
    yaxis=dict(showticklabels=False, zeroline=False, showgrid=False, title=""),
    legend=dict(title="<b>Tópicos</b>", orientation="v", y=1, x=1.02)
)

fig.write_html(GRAFICOS_DIR / "03_espacio_documentos.html")
fig.show()